In [72]:
import csv
import pandas # this is used to read the dataset
import time # this will be used to track the training time of the model
import torch # this will be the underlying framework of the model
import torch.amp # this will be used for faster processing on the gpu
from torch_geometric.nn import GCNConv, global_mean_pool # this will be used to create 
# the architecture of the model
from torch_geometric.data import Data, Batch # Data acts like the training data while Batch allows for a higher amount of data
# to be passed to the model
from torch.utils.data import DataLoader, Dataset # this allows the creation and loading
# of the dataset
import rdkit.Chem # this will be used to convert the molecular formula (in the form of
# a SMILES) into a molecule for further processing

In [73]:
if torch.cuda.is_available():
    train_on = 'cuda' # this will use the gpu if possible
    scalar = torch.amp.GradScaler() # this is used to target AI cores on the gpu
else:
    train_on = 'cpu' # if the gpu isn't available, use the cpu

In [74]:
h_receptors = ['NR-AR', 'NR-AR-LBD', 'NR-AhR',
               'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
               'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
               'SR-HSE', 'SR-MMP', 'SR-p53'] # this defines the 12 human receptors which the model will have to predict.

In [75]:

class Tox21Dataset(Dataset): # this creates a child class which inherits from the Dataset
    # class
    def __init__(self, csv_path):
        super().__init__()
        try:
            read_csv = pandas.read_csv(csv_path)
            self.dataframe = pandas.DataFrame(read_csv)
            print(self.dataframe[['NR-AR', 'NR-AR-LBD', 'NR-AhR',
                            'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
                            'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5',
                            'SR-HSE', 'SR-MMP', 'SR-p53', 'smiles']])
        except Exception as e:
            print(f"'{csv_path}' could not be read!")
    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        smiles = str(row['smiles'])

        mol = rdkit.Chem.MolFromSmiles(smiles)

        input = torch.zeros(len(mol.GetAtoms()), 118, dtype=torch.float) # this creates a 118x118 matrix of 0. When an atom/element is accessed,
        # it fills a specific row with a value, which represents that specific atom. It does the same for each other atom in the list, also
        # ensuring that each new element is given a separate column to properly represent all 118 elements in the periodic table.

        for atom in mol.GetAtoms():
            atomic_number = atom.GetAtomicNum() - 1 # subtract one to preform 0-indexing (counting from 0 instead of 1)
            if 0 <= atomic_number <= 118:
                input[atomic_number]

        molecule_index_list = []
        for molecule in mol.GetBonds():
            beginning_of_molecule = molecule.GetBeginAtomIdx()
            ending_of_molecule = molecule.GetEndAtomIdx()
            if isinstance(beginning_of_molecule, int) and isinstance(ending_of_molecule, int): # if there is an actual value for the beginning 
                #
                molecule_index_list.append([beginning_of_molecule, ending_of_molecule])

        molecule = torch.tensor([molecule_index_list], dtype=torch.long) # this creates a matrix which represents the molecule

        labels = [row.get(receptor) for receptor in h_receptors]

        if any(label not in (1, 0) for label in labels): # if any value is unknown in the dataset...
            return None # skip them

        true_labels = torch.tensor([labels], dtype=torch.long) # this creates a matrix which represents the actual receptors
        # a molecule may/may not bind to.

        data = Data(input=input, molecule=molecule, true_labels=true_labels)
        data.to(train_on) 
        return data

In [76]:
dataset = Tox21Dataset('tox21.csv')
dataset = [data for data in dataset if not None] # this filters any information in the data if it contains None (or NaN/Not a Number) values

def collate(batch):
    return Batch.from_data_list(batch)

loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate) # this loads the dataset with 64 shuffled molecules to train at a time 

      NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0       0.0        0.0     1.0           NaN    NaN        0.0            0.0   
1       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2       NaN        NaN     NaN           NaN    NaN        NaN            NaN   
3       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
4       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
...     ...        ...     ...           ...    ...        ...            ...   
7826    NaN        NaN     NaN           NaN    NaN        NaN            NaN   
7827    1.0        1.0     0.0           0.0    1.0        0.0            NaN   
7828    1.0        1.0     0.0           0.0    1.0        1.0            0.0   
7829    1.0        1.0     0.0           NaN    1.0        1.0            0.0   
7830    0.0        0.0     NaN           0.0    0.0        0.0            0.0   

      SR-ARE  SR-ATAD5  SR-

In [77]:
class Model(torch.nn.Module):
    def __init__(self, input_dimension, hidden_dimension, h_receptors):
        super().__init__()
        self.conv = GCNConv(input_dimension, hidden_dimension) # this will act as the backbone which will represent the molecular structure of
        # a given atom using the input_dimension as the input and outputting the output to the hidden dimension
        self.heads = torch.nn.ModuleList([torch.nn.Linear(hidden_dimension, 1) for _ in range(h_receptors)]) # for every receptor of a molecule, 
        # make one prediction.

    def forward_pass(self, data):
        input = data.input # equates a variable, input, with the inputs from the data (as defined two code block above)
        edges = data.edges # does the same thing except it used edges from data
        input = self.conv(input, edges) # this combines the inputs (the molecular characteristics and edges) to create a graph

        input = global_mean_pool(input, data.batch) # as it is, the input is the graphs/representations of the molecules. Using global_mean_pool
        # helps to "stack" these graphs together without losing their specific structures and characteristics
        logits = torch.cat([head(input) for head in self.heads], dim=1).squeeze(-1) # concatenate each of the model's predictions while returning a 
        # a scalar value (a tensor/matrix with 0 dimensions, i.e. a matrix with only one value)
        return logits # this returns whatever the value of logits is

In [78]:
model = Model(input_dimension=118, hidden_dimension=64, h_receptors=len(h_receptors)) # this sets the parameters of the model
model.to(train_on) # this moves the model to the gpu for training

criterion = torch.nn.BCEWithLogitsLoss() # this is set up to measure the model's accuracy (or rather, lack of it)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # this is set up to adjust the model's settings to make better predictions

training_loops = 10
for loop in range(training_loops):
    model.train() # this sets the model to training mode
    training_start_time = time.time() # this starts tracking the time of the training
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad() # reset the gradients to begin a new round of training (like clearing up any old information before starting something new)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            output = model(input) # this is the model's prediction

            loss = criterion(output, batch.true_labels)
        scalar.scale(loss).backward()
        scalar.step(optimizer)
        scalar.update()
        total_loss += loss.item()
        training_end_time = time.time()
        print(f"Total loss: {total_loss}, Batch: {batch}, Time: {training_start_time-training_end_time:.4f}")

print("Training Complete!")

TypeError: type 'NoneType' is not an acceptable base type